<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Crypto Foundations
## Chapter 3 – Hash Functions and Merkle Trees

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


This first exercise shows how to compute hashes of both in-memory messages and files. It mirrors the chapter’s message that hashes are low-level fingerprints you can attach to any byte stream to check for accidental corruption or tampering later on.

In [ ]:
import hashlib  # standard library hashing
from pathlib import Path  # convenient path handling

message = b"hash this message"  # in-memory bytes to hash
digest = hashlib.sha256(message).hexdigest()  # SHA-256 fingerprint
print("Message digest:", digest)

# For a self-contained lab, create a tiny file and hash its contents
path = Path("example_hash_file.txt")
if not path.exists():  # write the file once
    path.write_text("example file contents", encoding="utf-8")

h = hashlib.sha256()  # incremental hasher
with path.open("rb") as f:  # open file in binary mode
    for chunk in iter(lambda: f.read(4096), b""):
        h.update(chunk)  # feed chunks into the hash
print("File digest:", h.hexdigest())


Next you explore collisions in a deliberately tiny hash space. By truncating digests to just a few hex characters, collisions become common enough to observe with a short loop, reinforcing why real-world hashes use much larger outputs.

In [ ]:
import hashlib
import random  # to generate random messages
import string  # for simple random text

def random_message(length: int = 8) -> bytes:
    """Generate a random ASCII message as bytes."""
    alphabet = string.ascii_lowercase
    s = "".join(random.choice(alphabet) for _ in range(length))
    return s.encode("ascii")

seen = {}  # map from truncated digest to original message
trunc_len = 4  # hex characters to keep (tiny space!)

for i in range(10_000):  # cap attempts for this toy search
    msg = random_message()  # draw a random message
    full = hashlib.sha256(msg).hexdigest()  # full SHA-256 digest
    short = full[:trunc_len]  # truncated digest
    if short in seen and seen[short] != msg:
        print("Collision found after", i, "attempts")
        print("digest:", short)
        print("message 1:", seen[short])
        print("message 2:", msg)
        break
    seen[short] = msg  # record first message for this digest
else:
    print("No collision found in this run")


Finally, you build a tiny Merkle tree over a handful of leaf hashes. This exercise makes the abstract idea of "hashing pairs of hashes" concrete and shows how a single root can commit to many underlying records.

In [ ]:
import hashlib

def sha256_hex(data: bytes) -> str:
    """Helper to compute a hex-encoded SHA-256 hash."""
    return hashlib.sha256(data).hexdigest()

leaves = [b"tx1", b"tx2", b"tx3", b"tx4"]  # toy transaction payloads
leaf_hashes = [sha256_hex(m) for m in leaves]  # hash each leaf
print("Leaf hashes:")
for h in leaf_hashes:
    print(" ", h)

level = leaf_hashes[:]  # start from leaves
while len(level) > 1:  # build up until a single root remains
    next_level = []
    for i in range(0, len(level), 2):  # step over pairs
        left = level[i]
        right = level[i + 1] if i + 1 < len(level) else level[i]  # duplicate last if odd
        combined = bytes.fromhex(left) + bytes.fromhex(right)
        parent = sha256_hex(combined)  # parent hash
        next_level.append(parent)
    level = next_level

merkle_root = level[0]
print("Merkle root:", merkle_root)


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
